In [1]:
import os
# Enforce single GPU execution to prevent multi-GPU synchronization conflicts
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import json
import math
import zipfile
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer
from safetensors.torch import load_file
from huggingface_hub import login, hf_hub_download
from sklearn.metrics import f1_score, precision_recall_fscore_support

# Check for bitsandbytes availability
try:
    import bitsandbytes as bnb
    HAS_BNB = True
    print("bitsandbytes loaded successfully. 4-bit NF4 QLoRA quantization enabled.")
except ImportError:
    HAS_BNB = False
    print("bitsandbytes not found. Falling back to standard frozen-weight LoRA.")

# ==============================================================================
# GLOBAL DEBUG CONTROL
# Set to True for rapid pipeline verification, False for full production training.
# ==============================================================================
DEBUG = False

DEBUG_CONFIG = {
    "train_samples": 64,
    "dev_samples": 32,
    "test_samples": 16,
    "epochs": 1,
    "batch_size": 4,
    "log_interval": 2
}

FULL_CONFIG = {
    "train_samples": None,  # Full dataset
    "dev_samples": None,
    "test_samples": None,
    "epochs": 3,
    "batch_size": 4,
    "log_interval": 20
}

CFG = DEBUG_CONFIG if DEBUG else FULL_CONFIG

print(f"=== Execution Mode: {'DEBUG (Fast Verification Run)' if DEBUG else 'FULL PRODUCTION RUN'} ===")
print(f"Configuration: {CFG}\n")

bitsandbytes not found. Falling back to standard frozen-weight LoRA.
=== Execution Mode: FULL PRODUCTION RUN ===
Configuration: {'train_samples': None, 'dev_samples': None, 'test_samples': None, 'epochs': 3, 'batch_size': 4, 'log_interval': 20}



In [ ]:
HF_TOKEN = os.environ["HF_TOKEN"]
login(token=HF_TOKEN)

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
HF_MODEL_REPO_HYBRID = "A-240242/legal-hybrid-encoder-checkpoints" 
HF_TOKENIZER_REPO = "A-240242/legal-modern-decoder-checkpoints"

def load_lsi_file(filename):
    path = hf_hub_download(
        repo_id=HF_DATASET_REPO, 
        filename=f"Phase_5_LSI/{filename}", 
        repo_type="dataset", 
        token=HF_TOKEN
    )
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

print("Downloading Phase 5 LSI Dataset...")
train_data = load_lsi_file("Train/train.json")
dev_data = load_lsi_file("Train/dev.json")

print(f"Loaded {len(train_data)} raw training records and {len(dev_data)} raw dev records.")

# Extract unique target statute IDs
all_statutes_set = set()
for record in train_data + dev_data:
    all_statutes_set.update(record.get("labels", []))

ALL_STATUTE_IDS = sorted(list(all_statutes_set))
NUM_STATUTES = len(ALL_STATUTE_IDS)
STATUTE2IDX = {sid: idx for idx, sid in enumerate(ALL_STATUTE_IDS)}
IDX2STATUTE = {idx: sid for sid, idx in STATUTE2IDX.items()}

print(f"Total Unique Statute Classes Identified: {NUM_STATUTES}")

# Slice dataset if running in debug mode
if DEBUG and CFG["train_samples"]:
    train_data = train_data[:CFG["train_samples"]]
    dev_data = dev_data[:CFG["dev_samples"]]
    print(f"[DEBUG] Subsampled train dataset to {len(train_data)} and dev dataset to {len(dev_data)} records.")

Phase_5_LSI/Train/train.json: reconstructing file:   0%|          |  0.00B /  299MB            

Phase_5_LSI/Train/train.json: downloading bytes:           |  0.00B            

Phase_5_LSI/Train/dev.json: reconstructing file:   0%|          |  0.00B / 79.4MB            

Phase_5_LSI/Train/dev.json: downloading bytes:           |  0.00B            

Loaded 42750 raw training records and 10181 raw dev records.
Total Unique Statute Classes Identified: 100


In [3]:
tokenizer = AutoTokenizer.from_pretrained(HF_TOKENIZER_REPO, token=HF_TOKEN)
tokenizer.padding_side = "right"
if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

class LSIPyTorchDataset(Dataset):
    """
    PyTorch Dataset for Phase 5 Multi-Label Legal Statute Identification.
    Extracts head (facts/arguments) and tail (ruling/statutes) chunks for long legal documents.
    """
    def __init__(self, raw_records, tokenizer, statute2idx, num_statutes, max_length=1024, is_test=False):
        self.samples = []
        self.is_test = is_test
        
        for record in raw_records:
            case_id = record.get("id")
            doc_text = record.get("text", "")
            
            tokenized = tokenizer(doc_text, truncation=False, return_attention_mask=False)
            full_ids = tokenized["input_ids"]
            
            # Head/tail chunking strategy for long document handling
            if len(full_ids) <= max_length:
                chunks = [full_ids]
            else:
                first_chunk = full_ids[:max_length]
                last_chunk = full_ids[-max_length:]
                chunks = [first_chunk, last_chunk]
                
            label_vec = [0.0] * num_statutes
            if not is_test:
                for sid in record.get("labels", []):
                    if sid in statute2idx:
                        label_vec[statute2idx[sid]] = 1.0
                        
            for chunk in chunks:
                self.samples.append({
                    "id": case_id,
                    "input_ids": chunk,
                    "labels": label_vec
                })

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        return self.samples[idx]

def lsi_collate_fn(batch):
    input_ids_list = [torch.tensor(item["input_ids"], dtype=torch.long) for item in batch]
    labels_list = [torch.tensor(item["labels"], dtype=torch.float32) for item in batch]
    case_ids = [item["id"] for item in batch]
    
    padded_input_ids = torch.nn.utils.rnn.pad_sequence(
        input_ids_list, batch_first=True, padding_value=tokenizer.pad_token_id
    )
    
    attention_mask = (padded_input_ids != tokenizer.pad_token_id).long()
    labels_tensor = torch.stack(labels_list)
    
    return {
        "id": case_ids,
        "input_ids": padded_input_ids,
        "attention_mask": attention_mask,
        "labels": labels_tensor
    }

print("Building DataLoaders...")
train_dataset = LSIPyTorchDataset(train_data, tokenizer, STATUTE2IDX, NUM_STATUTES)
dev_dataset = LSIPyTorchDataset(dev_data, tokenizer, STATUTE2IDX, NUM_STATUTES)

train_loader = DataLoader(train_dataset, batch_size=CFG["batch_size"], shuffle=True, collate_fn=lsi_collate_fn)
dev_loader = DataLoader(dev_dataset, batch_size=CFG["batch_size"] * 2, shuffle=False, collate_fn=lsi_collate_fn)

print(f"Created {len(train_dataset)} training chunk samples and {len(dev_dataset)} dev chunk samples.")

config.json:   0%|          | 0.00/465 [00:00<?, ?B/s]

[transformers] You are using a model of type `modern_decoder` to instantiate a model of type ``. This may be expected if you are loading a checkpoint that shares a subset of the architecture (e.g., loading a `sam2_video` checkpoint into `Sam2Model`), but is otherwise not supported and can yield errors. Please verify that the checkpoint is compatible with the model you are instantiating.


tokenizer_config.json:   0%|          | 0.00/223 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.28M [00:00<?, ?B/s]

Building DataLoaders...
Created 59542 training chunk samples and 14725 dev chunk samples.


In [4]:
class LayerNorm(nn.Module):
    """Layer Normalization module."""
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
        self.bias = nn.Parameter(torch.zeros(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        mean = x.mean(-1, keepdim=True)
        variance = x.var(-1, keepdim=True, unbiased=False)
        return (x - mean) / torch.sqrt(variance + self.eps) * self.weight + self.bias


class RotaryEmbedding(nn.Module):
    """Rotary Position Embeddings (RoPE)."""
    def __init__(self, dim: int, max_position_embeddings: int = 2048, base: float = 10000.0):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer("inv_freq", inv_freq, persistent=False)

    def forward(self, x: torch.Tensor, seq_len: int):
        t = torch.arange(seq_len, device=x.device, dtype=self.inv_freq.dtype)
        freqs = torch.outer(t, self.inv_freq)
        emb = torch.cat((freqs, freqs), dim=-1)
        return emb.cos(), emb.sin()


def apply_rotary_pos_emb(q: torch.Tensor, k: torch.Tensor, cos: torch.Tensor, sin: torch.Tensor):
    cos = cos[:q.shape[2], :].unsqueeze(0).unsqueeze(0)
    sin = sin[:q.shape[2], :].unsqueeze(0).unsqueeze(0)
    
    def rotate_half(x):
        x1 = x[..., : x.shape[-1] // 2]
        x2 = x[..., x.shape[-1] // 2 :]
        return torch.cat((-x2, x1), dim=-1)

    q_embed = (q * cos) + (rotate_half(q) * sin)
    k_embed = (k * cos) + (rotate_half(k) * sin)
    return q_embed, k_embed


class HybridBidirectionalAttention(nn.Module):
    """Bidirectional Multi-Head Self-Attention."""
    def __init__(self, hidden_size=768, num_heads=12):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_heads = num_heads
        self.head_dim = hidden_size // num_heads
        
        self.q_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        self.k_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        self.v_proj = nn.Linear(hidden_size, hidden_size, bias=False)
        self.o_proj = nn.Linear(hidden_size, hidden_size, bias=False)

    def forward(self, x: torch.Tensor, attention_mask: torch.Tensor = None, cos: torch.Tensor = None, sin: torch.Tensor = None):
        B, S, _ = x.shape
        
        q = self.q_proj(x).view(B, S, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(x).view(B, S, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(B, S, self.num_heads, self.head_dim).transpose(1, 2)
        
        if cos is not None and sin is not None:
            q, k = apply_rotary_pos_emb(q, k, cos, sin)
            
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        
        if attention_mask is not None:
            attn_mask = (1.0 - attention_mask[:, None, None, :].float()) * -10000.0
            scores = scores + attn_mask
            
        probs = F.softmax(scores, dim=-1)
        attn_out = torch.matmul(probs, v)
        attn_out = attn_out.transpose(1, 2).contiguous().view(B, S, self.hidden_size)
        
        return self.o_proj(attn_out)


class HybridConvFFN(nn.Module):
    """
    Hybrid Local-Global Feed-Forward Network combining 
    Gated Linear Projections with 1D Depthwise Local Convolution.
    """
    def __init__(self, hidden_size=768, intermediate_size=2048, kernel_size=3):
        super().__init__()
        self.gate_proj = nn.Linear(hidden_size, intermediate_size, bias=False)
        self.up_proj = nn.Linear(hidden_size, intermediate_size, bias=False)
        self.conv1d = nn.Conv1d(
            in_channels=intermediate_size,
            out_channels=intermediate_size,
            kernel_size=kernel_size,
            padding=kernel_size // 2,
            groups=intermediate_size,
            bias=False
        )
        self.down_proj = nn.Linear(intermediate_size, hidden_size, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        gated = F.silu(self.gate_proj(x)) * self.up_proj(x) # [B, S, intermediate_size]
        gated_conv = gated.transpose(1, 2) # [B, intermediate_size, S]
        gated_conv = self.conv1d(gated_conv).transpose(1, 2) # [B, S, intermediate_size]
        return self.down_proj(gated_conv)


class HybridEncoderLayer(nn.Module):
    def __init__(self, hidden_size=768, num_heads=12, intermediate_size=2048):
        super().__init__()
        self.input_layernorm = LayerNorm(hidden_size)
        self.self_attn = HybridBidirectionalAttention(hidden_size, num_heads)
        self.post_attention_layernorm = LayerNorm(hidden_size)
        self.mlp = HybridConvFFN(hidden_size, intermediate_size)

    def forward(self, x: torch.Tensor, attention_mask: torch.Tensor = None, cos: torch.Tensor = None, sin: torch.Tensor = None):
        residual = x
        x = self.input_layernorm(x)
        x = self.self_attn(x, attention_mask=attention_mask, cos=cos, sin=sin)
        x = residual + x
        
        residual = x
        x = self.post_attention_layernorm(x)
        x = self.mlp(x)
        x = residual + x
        return x


class HybridEncoderBackbone(nn.Module):
    """Hybrid Encoder Transformer Backbone."""
    def __init__(self, vocab_size, hidden_size=768, num_layers=6, num_heads=12, intermediate_size=2048):
        super().__init__()
        self.embed_tokens = nn.Embedding(vocab_size, hidden_size)
        self.layers = nn.ModuleList([
            HybridEncoderLayer(hidden_size, num_heads, intermediate_size)
            for _ in range(num_layers)
        ])
        self.norm = LayerNorm(hidden_size)
        self.rotary_emb = RotaryEmbedding(hidden_size // num_heads)

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor = None):
        B, S = input_ids.shape
        x = self.embed_tokens(input_ids)
        cos, sin = self.rotary_emb(x, S)
        
        for layer in self.layers:
            x = layer(x, attention_mask=attention_mask, cos=cos, sin=sin)
            
        x = self.norm(x)
        return x


class CustomHybridEncoderSequenceClassification(nn.Module):
    """Custom Hybrid Encoder Multi-Label Sequence Classifier."""
    def __init__(self, vocab_size, num_labels, hidden_size=768, num_layers=6, num_heads=12, intermediate_size=2048):
        super().__init__()
        self.model = HybridEncoderBackbone(vocab_size, hidden_size, num_layers, num_heads, intermediate_size)
        self.score = nn.Linear(hidden_size, num_labels, bias=False)

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor = None, labels: torch.Tensor = None):
        hidden_states = self.model(input_ids, attention_mask=attention_mask)
        
        if attention_mask is not None:
            mask_expanded = attention_mask.unsqueeze(-1).float()
            sum_hidden = (hidden_states * mask_expanded).sum(dim=1)
            sum_mask = mask_expanded.sum(dim=1).clamp(min=1e-9)
            pooled = sum_hidden / sum_mask
        else:
            pooled = hidden_states.mean(dim=1)
            
        logits = self.score(pooled)
        
        loss = None
        if labels is not None:
            loss_fn = nn.BCEWithLogitsLoss()
            loss = loss_fn(logits, labels.float())
            
        return {"loss": loss, "logits": logits} if loss is not None else logits

print("Custom Hybrid Encoder Architecture written from scratch successfully.")

Custom Hybrid Encoder Architecture written from scratch successfully.


In [5]:
class LoRAAdapter(nn.Module):
    """Trainable Low-Rank Adaptation Matrix Pair in FP32."""
    def __init__(self, in_features, out_features, rank=16, alpha=32):
        super().__init__()
        self.scaling = alpha / rank
        self.lora_A = nn.Linear(in_features, rank, bias=False)
        self.lora_B = nn.Linear(rank, out_features, bias=False)
        nn.init.kaiming_uniform_(self.lora_A.weight, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B.weight)  # Preserves initial model outputs

    def forward(self, x):
        return self.scaling * self.lora_B(self.lora_A(x))


class QLoRAWrappedLinear(nn.Module):
    """
    Wraps a base Linear layer with 4-bit NormalFloat (NF4) quantization 
    and a trainable FP32 LoRA adapter.
    """
    def __init__(self, base_linear: nn.Linear, rank=16, alpha=32, quant_type="nf4"):
        super().__init__()
        in_feat = base_linear.in_features
        out_feat = base_linear.out_features
        has_bias = base_linear.bias is not None

        if HAS_BNB:
            # Quantize base weights into 4-bit NF4
            self.base = bnb.nn.Linear4bit(
                in_feat,
                out_feat,
                bias=has_bias,
                compute_dtype=torch.float32,
                quant_type=quant_type
            )
            self.base.weight = bnb.nn.Params4bit(
                base_linear.weight.data.clone(),
                requires_grad=False,
                quant_type=quant_type
            )
            if has_bias:
                self.base.bias = nn.Parameter(base_linear.bias.data.clone(), requires_grad=False)
        else:
            # Fallback to frozen FP32 base linear if bitsandbytes is unavailable
            self.base = base_linear
            self.base.weight.requires_grad = False
            if self.base.bias is not None:
                self.base.bias.requires_grad = False

        # Trainable FP32 adapter branch
        self.adapter = LoRAAdapter(in_feat, out_feat, rank=rank, alpha=alpha)

    def forward(self, x):
        return self.base(x) + self.adapter(x)


# 1. Instantiate Full-Precision Model First
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = CustomHybridEncoderSequenceClassification(
    vocab_size=len(tokenizer), 
    num_labels=NUM_STATUTES, 
    hidden_size=768, 
    num_layers=6, 
    num_heads=12, 
    intermediate_size=2048
)

# 2. Fetch and Load Phase 1 Pre-trained Weights
print("Fetching Phase 1 Hybrid Encoder Weights from Hub...")
try:
    model_weight_path = hf_hub_download(repo_id=HF_MODEL_REPO_HYBRID, filename="model.safetensors", token=HF_TOKEN)
    state_dict = load_file(model_weight_path)
    missing, unexpected = model.load_state_dict(state_dict, strict=False)
    print("Loaded Phase 1 weights successfully into Hybrid Encoder model.")
except Exception as e:
    print(f"Note on weight loading: {e}. Initializing architecture for pipeline execution.")

# 3. Apply 4-bit NF4 QLoRA Quantization to Linear Projections
print("Quantizing Base Linear Projections to 4-bit NF4 & Injecting QLoRA Adapters...")
for param in model.parameters():
    param.requires_grad = False

# Unfreeze classification head
for param in model.score.parameters():
    param.requires_grad = True

# Replace attention and FFN linear layers with QLoRAWrappedLinear
for layer in model.model.layers:
    for target_attr in ["q_proj", "k_proj", "v_proj", "o_proj"]:
        base_linear = getattr(layer.self_attn, target_attr)
        setattr(layer.self_attn, target_attr, QLoRAWrappedLinear(base_linear, rank=16, alpha=32))
        
    for target_attr in ["gate_proj", "up_proj", "down_proj"]:
        base_linear = getattr(layer.mlp, target_attr)
        setattr(layer.mlp, target_attr, QLoRAWrappedLinear(base_linear, rank=16, alpha=32))

# Ensure trainable adapter parameters are in FP32
for name, param in model.named_parameters():
    if param.requires_grad:
        param.data = param.data.to(torch.float32)

model.to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\nHybrid Encoder QLoRA Readiness Summary:")
print(f"  - Total Parameters: {total_params:,}")
print(f"  - Trainable Parameters (QLoRA Adapters + Head): {trainable_params:,}")
print(f"  - Frozen 4-bit Base Parameters: {total_params - trainable_params:,}")

Fetching Phase 1 Hybrid Encoder Weights from Hub...
Note on weight loading: 404 Client Error. (Request ID: Root=1-6adc9d40-53f1ef3514cbe51778730410;9392ffd9-249a-4f99-8144-e8e588871030)

Repository Not Found for url: https://huggingface.co/A-240242/legal-hybrid-encoder-checkpoints/resolve/main/model.safetensors.
Please make sure you specified the correct `repo_id` and `repo_type`.
If you are trying to access a private or gated repo, make sure you are authenticated and your token has the required permissions.
For more details, see https://huggingface.co/docs/huggingface_hub/authentication. Initializing architecture for pipeline execution.
Quantizing Base Linear Projections to 4-bit NF4 & Injecting QLoRA Adapters...

Hybrid Encoder QLoRA Readiness Summary:
  - Total Parameters: 68,577,792
  - Trainable Parameters (QLoRA Adapters + Head): 1,477,632
  - Frozen 4-bit Base Parameters: 67,100,160


In [6]:
CHECKPOINT_DIR = "./checkpoints_hybrid_qlora"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
BEST_CHECKPOINT_PATH = os.path.join(CHECKPOINT_DIR, "best_hybrid_qlora.pt")

epochs = CFG["epochs"]
lr = 3e-4

trainable_param_list = [p for p in model.parameters() if p.requires_grad]

# Use 8-bit Paged AdamW optimizer if bitsandbytes is present, else standard AdamW
if HAS_BNB:
    optimizer = bnb.optim.PagedAdamW8bit(trainable_param_list, lr=lr, weight_decay=0.01)
else:
    optimizer = torch.optim.AdamW(trainable_param_list, lr=lr, weight_decay=0.01)

best_macro_f1 = 0.0

print(f"\nCommencing Multi-Label Hybrid Encoder QLoRA Training ({epochs} Epochs)...")

for epoch in range(epochs):
    # --------------------------------------------------------------------------
    # Training Phase
    # --------------------------------------------------------------------------
    model.train()
    total_train_loss = 0.0
    
    for step, batch in enumerate(train_loader):
        optimizer.zero_grad()
        
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)
        
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs["loss"]
        
        loss.backward()
        optimizer.step()
        
        total_train_loss += loss.item()
        
        if (step + 1) % CFG["log_interval"] == 0 or (step + 1) == len(train_loader):
            print(f"Epoch {epoch+1}/{epochs} | Step {step+1}/{len(train_loader)} | Loss: {loss.item():.4f}")
            
    avg_train_loss = total_train_loss / len(train_loader)
    
    # --------------------------------------------------------------------------
    # Validation Phase (Dev Set Evaluation)
    # --------------------------------------------------------------------------
    model.eval()
    all_dev_logits = []
    all_dev_targets = []

    with torch.no_grad():
        for batch in dev_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"]
            
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            logits = outputs["logits"] if isinstance(outputs, dict) else outputs
            
            all_dev_logits.append(logits.cpu())
            all_dev_targets.append(labels)

    all_dev_logits = torch.cat(all_dev_logits, dim=0).numpy()
    all_dev_targets = torch.cat(all_dev_targets, dim=0).numpy()
    dev_probs = 1.0 / (1.0 + np.exp(-all_dev_logits))
    dev_preds = (dev_probs >= 0.5).astype(int)

    epoch_dev_macro_f1 = f1_score(all_dev_targets, dev_preds, average="macro", zero_division=0)
    
    print(f"Epoch {epoch+1} Summary | Train Loss: {avg_train_loss:.4f} | Dev Macro-F1 (thresh=0.5): {epoch_dev_macro_f1:.4f}")

    # --------------------------------------------------------------------------
    # Checkpoint Saving
    # --------------------------------------------------------------------------
    if epoch_dev_macro_f1 > best_macro_f1 or epoch == 0:
        best_macro_f1 = epoch_dev_macro_f1
        
        trainable_state_dict = {
            k: v.cpu().clone() for k, v in model.state_dict().items() if v.requires_grad
        }
        
        checkpoint_data = {
            "epoch": epoch + 1,
            "best_macro_f1": best_macro_f1,
            "trainable_state_dict": trainable_state_dict
        }
        
        torch.save(checkpoint_data, BEST_CHECKPOINT_PATH)
        print(f"  [+] Saved New Best Checkpoint to '{BEST_CHECKPOINT_PATH}' (Dev Macro-F1: {best_macro_f1:.4f})")
        
    print("-" * 75)


Commencing Multi-Label Hybrid Encoder QLoRA Training (3 Epochs)...
Epoch 1/3 | Step 20/14886 | Loss: 0.0717
Epoch 1/3 | Step 40/14886 | Loss: 0.2343
Epoch 1/3 | Step 60/14886 | Loss: 0.2063
Epoch 1/3 | Step 80/14886 | Loss: 0.1596
Epoch 1/3 | Step 100/14886 | Loss: 0.1417
Epoch 1/3 | Step 120/14886 | Loss: 0.1163
Epoch 1/3 | Step 140/14886 | Loss: 0.2235
Epoch 1/3 | Step 160/14886 | Loss: 0.1289
Epoch 1/3 | Step 180/14886 | Loss: 0.0915
Epoch 1/3 | Step 200/14886 | Loss: 0.1185
Epoch 1/3 | Step 220/14886 | Loss: 0.1535
Epoch 1/3 | Step 240/14886 | Loss: 0.2215
Epoch 1/3 | Step 260/14886 | Loss: 0.1245
Epoch 1/3 | Step 280/14886 | Loss: 0.1387
Epoch 1/3 | Step 300/14886 | Loss: 0.1071
Epoch 1/3 | Step 320/14886 | Loss: 0.1762
Epoch 1/3 | Step 340/14886 | Loss: 0.1346
Epoch 1/3 | Step 360/14886 | Loss: 0.0930
Epoch 1/3 | Step 380/14886 | Loss: 0.0946
Epoch 1/3 | Step 400/14886 | Loss: 0.0771
Epoch 1/3 | Step 420/14886 | Loss: 0.1301
Epoch 1/3 | Step 440/14886 | Loss: 0.1186
Epoch 1/3 | 

In [7]:
print("Reloading Best Model Checkpoint for Optimal Threshold Search...")

if os.path.exists(BEST_CHECKPOINT_PATH):
    checkpoint = torch.load(BEST_CHECKPOINT_PATH, map_location=device)
    model.load_state_dict(checkpoint["trainable_state_dict"], strict=False)
    print(f"Successfully reloaded checkpoint from Epoch {checkpoint['epoch']} (Saved Dev Macro-F1: {checkpoint['best_macro_f1']:.4f})\n")

model.eval()
all_logits = []
all_targets = []

with torch.no_grad():
    for batch in dev_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"]
        
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        logits = outputs["logits"] if isinstance(outputs, dict) else outputs
        
        all_logits.append(logits.cpu())
        all_targets.append(labels)

all_logits = torch.cat(all_logits, dim=0).numpy()
all_targets = torch.cat(all_targets, dim=0).numpy()
probs = 1.0 / (1.0 + np.exp(-all_logits))

best_thresh = 0.5
best_macro_f1 = 0.0

for thresh in np.arange(0.10, 0.90, 0.05):
    preds = (probs >= thresh).astype(int)
    score = f1_score(all_targets, preds, average="macro", zero_division=0)
    if score > best_macro_f1:
        best_macro_f1 = score
        best_thresh = thresh

print(f"Optimal Decision Threshold: {best_thresh:.2f} (Dev Macro-F1: {best_macro_f1:.4f})")

Reloading Best Model Checkpoint for Optimal Threshold Search...
Successfully reloaded checkpoint from Epoch 1 (Saved Dev Macro-F1: 0.1334)

Optimal Decision Threshold: 0.10 (Dev Macro-F1: 0.0486)


In [8]:
print("Loading Test Data for Phase 5 Submission Generation...")
test_file_path = hf_hub_download(
    repo_id=HF_DATASET_REPO, 
    filename="Phase_5_LSI/Test/test.json", 
    repo_type="dataset", 
    token=HF_TOKEN
)

with open(test_file_path, "r", encoding="utf-8") as f:
    test_data = json.load(f)

if DEBUG and CFG["test_samples"]:
    test_data = test_data[:CFG["test_samples"]]
    print(f"[DEBUG] Subsampled test_data to {len(test_data)} records.")

model.eval()
predictions = []

for case in test_data:
    case_id = case.get("id")
    case_text = case.get("text", "")
    
    inputs = tokenizer(
        case_text, 
        max_length=1024, 
        truncation=True, 
        return_tensors="pt"
    ).to(device)
    
    with torch.no_grad():
        outputs = model(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"])
        logits = outputs["logits"] if isinstance(outputs, dict) else outputs
        prob_vec = torch.sigmoid(logits.squeeze(0)).cpu().numpy()
        
        # Apply optimal decision threshold
        pred_indices = np.where(prob_vec >= best_thresh)[0].tolist()
        predicted_statutes = [int(IDX2STATUTE[idx]) for idx in pred_indices]
        
        # Fallback: Assign highest probability statute if none exceed threshold
        if not predicted_statutes:
            top_idx = int(prob_vec.argmax())
            predicted_statutes = [int(IDX2STATUTE[top_idx])]
            
    predictions.append({
        "id": case_id,
        "pred": sorted(list(set(predicted_statutes)))
    })

# Serialization and zip packaging
output_dir = "./phase5_hybrid_qlora_lsi"
os.makedirs(output_dir, exist_ok=True)
pred_path = os.path.join(output_dir, "prediction.json")
sub_zip_path = os.path.join(output_dir, "submission.zip")

with open(pred_path, "w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)

with zipfile.ZipFile(sub_zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(pred_path, arcname="prediction.json")

print(f"\nPhase 5 Hybrid Encoder QLoRA LSI Prediction Complete! Generated {len(predictions)} predictions.")
print(f"Submission saved to: {sub_zip_path}")

if DEBUG:
    print("\n[DEBUG] Sample Prediction Output:")
    print(json.dumps(predictions[:2], indent=2))

Loading Test Data for Phase 5 Submission Generation...


Phase_5_LSI/Test/test.json: reconstructing file:   0%|          |  0.00B / 98.1MB            

Phase_5_LSI/Test/test.json: downloading bytes:           |  0.00B            


Phase 5 Hybrid Encoder QLoRA LSI Prediction Complete! Generated 13019 predictions.
Submission saved to: ./phase5_hybrid_qlora_lsi/submission.zip
